In [1]:
import yaml
import joblib
import json
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_anthropic import ChatAnthropic
from prompt_templates import RAG_TOOL_PROMPT_PART2
from agent_tools import compare_dates
from langchain_core.messages import HumanMessage, ToolMessage

with open('secret.key') as key_file:
    api_key = key_file.readline().strip('\n').strip()

In [2]:
with open('config.yaml', 'rb') as yaml_file:
    config=yaml.safe_load(yaml_file)

In [3]:
embeddings = HuggingFaceEmbeddings(model_name='google/embeddinggemma-300m')

In [4]:
try:
    chroma_vectorstore = Chroma(
        collection_name=config['collection_name'],
        persist_directory=config['chroma_dir'],
        embedding_function=embeddings,
        create_collection_if_not_exists=False
    )
    dense_retriever = chroma_vectorstore.as_retriever(
        search_type='similarity',
        search_kwargs={'k': config['retrieval_k']},
    )
    with open(config['bm25_index_path'], 'rb') as bm25_file:
        sparse_retriever = joblib.load(bm25_file)

    hybrid_retriever = EnsembleRetriever(
        retrievers=[dense_retriever, sparse_retriever],
        weights=[config['vector_search_weightage'], (1-config['vector_search_weightage'])],
    )
except:
    print('[Error] Vector database or bm25 index missing, please run part1_1 script first')

In [5]:
llm = ChatAnthropic(
    # model_name='claude-haiku-4-5-20251001',
    # temperature=0.0,
    # # top_k=10,
    # # top_p=0.5,
    model_name='claude-sonnet-5',
    max_tokens=4096,
    timeout=30,
    max_retries=2,
    anthropic_api_key=api_key
)

llm_tool = llm.bind_tools([compare_dates])
print(llm_tool)

bound=ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-anthropic': '1.4.8'}}, output_version=None, model='claude-sonnet-5', max_tokens=4096, default_request_timeout=30.0, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), anthropic_proxy=None, model_kwargs={}) kwargs={'tools': [{'name': 'compare_dates', 'input_schema': {'properties': {'date_str': {'type': 'string'}, 'reference_date': {'default': '2024-01-01', 'type': 'string'}, 'date_format': {'default': '%Y-%m-%d', 'type': 'string'}}, 'required': ['date_str'], 'type': 'object'}, 'description': 'Compare date_str against reference_date and generate a label as \'Expired\' or \'Upcoming\' or \'Ongoing\' only\n\nArgs:\n    date_str: The input date for comparison, e.g. "2024-02-15".\n    reference_date: The date to compare against, e.g. "2024-01-01". Defaults to "2024-01-01".\n    date_format: The strptime format both dates are in. Defaults to "%Y-%m-

In [6]:
def rag_tool_invoke(query, retriever, prompt):
    rag_docs = retriever.invoke(query)
    context = '\n\n'.join(doc.page_content for doc in rag_docs)

    filled_prompt = prompt.format(
        context=context,
        input=query,
    )

    conversation = [HumanMessage(content=filled_prompt)]
    model_response = llm_tool.invoke(conversation)
    conversation.append(model_response)

    while model_response.tool_calls:
        for tool_call in model_response.tool_calls:
            tool_result = compare_dates.invoke(tool_call['args'])

            conversation.append(
                ToolMessage(content=json.dumps(tool_result), tool_call_id=tool_call['id'])
            )

        model_response = llm_tool.invoke(conversation)
        conversation.append(model_response)
    
    json_result = json.loads(model_response.content)
    return json_result

In [ ]:
query = "what is the document's distribution date"

output = rag_tool_invoke(
    query=query,
    retriever=hybrid_retriever,
    prompt=RAG_TOOL_PROMPT_PART2
)
print(output)

[{'original_text': 'Distributed on Budget Day: 16 February 2024', 'normalized_date': '2024-02-16', 'status': 'Upcoming'}]


In [8]:
query = "what is the date relating to estate duty"

output = rag_tool_invoke(
    query=query,
    retriever=hybrid_retriever,
    prompt=RAG_TOOL_PROMPT_PART2
)
print(output)

[{'original_text': 'Estate Duty does not apply to a person who dies after 15 February 2008.', 'normalized_date': '2008-02-15', 'status': 'Expired'}]


In [9]:
query = "what are the dates related to document distribution and estate duty"

output = rag_tool_invoke(
    query=query,
    retriever=hybrid_retriever,
    prompt=RAG_TOOL_PROMPT_PART2
)
print(output)

[{'original_text': 'Distributed on Budget Day: 16 February 2024', 'normalized_date': '2024-02-16', 'status': 'Upcoming'}, {'original_text': 'Estate Duty does not apply to a person who dies after 15 February 2008', 'normalized_date': '2008-02-15', 'status': 'Expired'}]


In [10]:
query = "what are the key dates to remember from this document"

output = rag_tool_invoke(
    query=query,
    retriever=hybrid_retriever,
    prompt=RAG_TOOL_PROMPT_PART2
)
print(output)

[{'original_text': 'Output gap estimated as at 5 January 2024', 'normalized_date': '2024-01-05', 'status': 'Upcoming'}, {'original_text': 'FY2024 Revenue and Expenditure Estimates presented to Parliament on 16 February 2024', 'normalized_date': '2024-02-16', 'status': 'Upcoming'}, {'original_text': 'increase in Property Tax rates which took effect on 1 January 2024', 'normalized_date': '2024-01-01', 'status': 'Ongoing'}]


In [11]:
# query = 'What are the key government revenue streams'
# query = 'how will the Budget for the Future Energy Fund be supported?'
# query = 'What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?'